In [6]:
# Cell 1
import os
import gc
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import log_loss
import lightgbm as lgb
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

BASE_DIR = Path("/mnt/c/etri-lifelog/data/raw/data")
SENSOR_DIR = BASE_DIR / "ch2025_data_items"

TRAIN_PATH = BASE_DIR / "ch2026_metrics_train.csv"
SUB_PATH = BASE_DIR / "ch2026_submission_sample.csv"

TARGETS = ["Q1", "Q2", "Q3", "S1", "S2", "S3", "S4"]

print(BASE_DIR)
print(SENSOR_DIR.exists(), TRAIN_PATH.exists(), SUB_PATH.exists())

/mnt/c/etri-lifelog/data/raw/data
True True True


In [7]:
# Cell 2
train = pd.read_csv(TRAIN_PATH)
sub = pd.read_csv(SUB_PATH)

train["subject_id"] = train["subject_id"].astype(str)
train["lifelog_date"] = pd.to_datetime(train["lifelog_date"])

if "subject_id" in sub.columns:
    sub["subject_id"] = sub["subject_id"].astype(str)
if "lifelog_date" in sub.columns:
    sub["lifelog_date"] = pd.to_datetime(sub["lifelog_date"])

print(train.shape, sub.shape)
display(train.head())
display(sub.head())

(450, 10) (250, 10)


,subject_id,sleep_date,lifelog_date,Q1,Q2,Q3,S1,S2,S3,S4
0,id01,2024-06-27,2024-06-26,0,0,0,0,0,1,0
1,id01,2024-06-28,2024-06-27,0,0,0,0,1,1,1
2,id01,2024-06-29,2024-06-28,1,0,0,1,1,1,1
3,id01,2024-06-30,2024-06-29,1,0,1,1,0,0,0
4,id01,2024-07-01,2024-06-30,0,1,1,1,1,1,1


,subject_id,sleep_date,lifelog_date,Q1,Q2,Q3,S1,S2,S3,S4
0,id01,2024-07-31,2024-07-30,0,0,0,0,0,0,0
1,id01,2024-08-01,2024-07-31,0,0,0,0,0,0,0
2,id01,2024-08-02,2024-08-01,0,0,0,0,0,0,0
3,id01,2024-08-03,2024-08-02,0,0,0,0,0,0,0
4,id01,2024-08-04,2024-08-03,0,0,0,0,0,0,0


In [8]:
# Cell 3
def infer_test_frame_from_submission(train_df, sub_df):
    if {"subject_id", "lifelog_date"}.issubset(sub_df.columns):
        test_df = sub_df[["subject_id", "lifelog_date"]].copy()
        test_df["subject_id"] = test_df["subject_id"].astype(str)
        test_df["lifelog_date"] = pd.to_datetime(test_df["lifelog_date"])
        return test_df
    
    n_test = len(sub_df)
    last_dates = train_df.groupby("subject_id")["lifelog_date"].max().sort_values()
    subs = list(last_dates.index)
    reps = math.ceil(n_test / len(subs))
    subject_seq = (subs * reps)[:n_test]

    next_dates = {sid: train_df.loc[train_df["subject_id"] == sid, "lifelog_date"].max() + pd.Timedelta(days=1) for sid in subs}
    rows = []
    for sid in subject_seq:
        rows.append((sid, next_dates[sid]))
        next_dates[sid] += pd.Timedelta(days=1)

    test_df = pd.DataFrame(rows, columns=["subject_id", "lifelog_date"])
    return test_df

test = infer_test_frame_from_submission(train, sub)
print(test.shape)
display(test.head())
display(test.tail())

(250, 2)


,subject_id,lifelog_date
0,id01,2024-07-30
1,id01,2024-07-31
2,id01,2024-08-01
3,id01,2024-08-02
4,id01,2024-08-03


,subject_id,lifelog_date
245,id10,2024-09-21
246,id10,2024-09-22
247,id10,2024-09-24
248,id10,2024-09-25
249,id10,2024-09-26


In [9]:
# Cell 4
base_df = pd.concat(
    [
        train[["subject_id", "lifelog_date"] + TARGETS].copy(),
        test[["subject_id", "lifelog_date"]].copy()
    ],
    axis=0,
    ignore_index=True
).sort_values(["subject_id", "lifelog_date"]).reset_index(drop=True)

base_df["dow"] = base_df["lifelog_date"].dt.dayofweek
base_df["month"] = base_df["lifelog_date"].dt.month
base_df["day"] = base_df["lifelog_date"].dt.day
base_df["is_weekend"] = (base_df["dow"] >= 5).astype(int)
base_df["days_from_global_start"] = (base_df["lifelog_date"] - base_df["lifelog_date"].min()).dt.days

display(base_df.head())

,subject_id,lifelog_date,Q1,Q2,Q3,S1,S2,S3,S4,dow,month,day,is_weekend,days_from_global_start
0,id01,2024-06-26,0.0,0.0,0.0,0.0,0.0,1.0,0.0,2,6,26,0,23
1,id01,2024-06-27,0.0,0.0,0.0,0.0,1.0,1.0,1.0,3,6,27,0,24
2,id01,2024-06-28,1.0,0.0,0.0,1.0,1.0,1.0,1.0,4,6,28,0,25
3,id01,2024-06-29,1.0,0.0,1.0,1.0,0.0,0.0,0.0,5,6,29,1,26
4,id01,2024-06-30,0.0,1.0,1.0,1.0,1.0,1.0,1.0,6,6,30,1,27


In [10]:
# Cell 5
def get_sensor_path(keyword):
    files = sorted(SENSOR_DIR.glob("*.parquet"))
    for p in files:
        if keyword.lower() in p.name.lower():
            return p
    return None

sensor_map = {
    "activity": get_sensor_path("mActivity"),
    "light": get_sensor_path("mLight"),
    "screen": get_sensor_path("mScreenStatus"),
    "hr": get_sensor_path("wHr"),
    "pedo": get_sensor_path("wPedo"),
    "charge": get_sensor_path("mACStatus"),
}

sensor_map

{'activity': PosixPath('/mnt/c/etri-lifelog/data/raw/data/ch2025_data_items/ch2025_mActivity.parquet'),
 'light': PosixPath('/mnt/c/etri-lifelog/data/raw/data/ch2025_data_items/ch2025_mLight.parquet'),
 'screen': PosixPath('/mnt/c/etri-lifelog/data/raw/data/ch2025_data_items/ch2025_mScreenStatus.parquet'),
 'hr': PosixPath('/mnt/c/etri-lifelog/data/raw/data/ch2025_data_items/ch2025_wHr.parquet'),
 'pedo': PosixPath('/mnt/c/etri-lifelog/data/raw/data/ch2025_data_items/ch2025_wPedo.parquet'),
 'charge': PosixPath('/mnt/c/etri-lifelog/data/raw/data/ch2025_data_items/ch2025_mACStatus.parquet')}

In [11]:
# Cell 6
def prep_sensor_df(df, value_cols):
    df = df.copy()
    df["subject_id"] = df["subject_id"].astype(str)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["lifelog_date"] = df["timestamp"].dt.floor("D")
    df["hour"] = df["timestamp"].dt.hour
    df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 6)).astype(int)
    df["is_day"] = ((df["hour"] >= 9) & (df["hour"] <= 18)).astype(int)
    keep = ["subject_id", "lifelog_date", "timestamp", "hour", "is_night", "is_day"] + [c for c in value_cols if c in df.columns]
    return df[keep].copy()

def add_simple_agg(df, val_col, prefix):
    g = df.groupby(["subject_id", "lifelog_date"])[val_col]
    feat = g.agg(["mean", "std", "min", "max", "count"]).reset_index()
    feat.columns = ["subject_id", "lifelog_date"] + [f"{prefix}_{c}" for c in ["mean", "std", "min", "max", "count"]]

    night = (
        df[df["is_night"] == 1]
        .groupby(["subject_id", "lifelog_date"])[val_col]
        .agg(["mean", "std"])
        .reset_index()
    )
    night.columns = ["subject_id", "lifelog_date", f"{prefix}_night_mean", f"{prefix}_night_std"]

    day = (
        df[df["is_day"] == 1]
        .groupby(["subject_id", "lifelog_date"])[val_col]
        .agg(["mean", "std"])
        .reset_index()
    )
    day.columns = ["subject_id", "lifelog_date", f"{prefix}_day_mean", f"{prefix}_day_std"]

    out = feat.merge(night, on=["subject_id", "lifelog_date"], how="left")
    out = out.merge(day, on=["subject_id", "lifelog_date"], how="left")
    return out

In [12]:
# Cell 7
def build_activity_features():
    p = sensor_map["activity"]
    if p is None:
        return None
    df = pd.read_parquet(p)
    df = prep_sensor_df(df, ["m_activity"])
    feat = add_simple_agg(df, "m_activity", "m_activity")
    return feat

def build_light_features():
    p = sensor_map["light"]
    if p is None:
        return None
    df = pd.read_parquet(p)
    df = prep_sensor_df(df, ["m_light"])
    feat = add_simple_agg(df, "m_light", "m_light")
    return feat

def build_screen_features():
    p = sensor_map["screen"]
    if p is None:
        return None
    df = pd.read_parquet(p)
    df = prep_sensor_df(df, ["m_screen_use"])
    feat = add_simple_agg(df, "m_screen_use", "m_screen_use")
    return feat

def build_charge_features():
    p = sensor_map["charge"]
    if p is None:
        return None
    df = pd.read_parquet(p)
    df = prep_sensor_df(df, ["m_charging"])
    feat = add_simple_agg(df, "m_charging", "m_charging")
    return feat

activity_feat = build_activity_features()
light_feat = build_light_features()
screen_feat = build_screen_features()
charge_feat = build_charge_features()

for name, feat in [("activity", activity_feat), ("light", light_feat), ("screen", screen_feat), ("charge", charge_feat)]:
    print(name, None if feat is None else feat.shape)

activity (700, 11)
light (700, 11)
screen (700, 11)
charge (700, 11)


In [ ]:
def explode_hr_array(x):
    # parquet에서 numpy array / list / object array / string(json) 다 처리
    if x is None:
        return []

    if isinstance(x, np.ndarray):
        if x.size == 0:
            return []
        return pd.to_numeric(pd.Series(x.ravel()), errors="coerce").dropna().tolist()

    if isinstance(x, (list, tuple)):
        if len(x) == 0:
            return []
        return pd.to_numeric(pd.Series(list(x)), errors="coerce").dropna().tolist()

    if isinstance(x, str):
        s = x.strip()
        if s == "":
            return []
        try:
            v = json.loads(s)
            if isinstance(v, np.ndarray):
                return pd.to_numeric(pd.Series(v.ravel()), errors="coerce").dropna().tolist()
            if isinstance(v, (list, tuple)):
                return pd.to_numeric(pd.Series(list(v)), errors="coerce").dropna().tolist()
            return []
        except:
            return []

    # 스칼라 처리
    try:
        if pd.isna(x):
            return []
    except:
        pass

    try:
        return [float(x)]
    except:
        return []

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [ ]:
# Cell 9
def build_pedo_features():
    p = sensor_map["pedo"]
    if p is None:
        return None

    df = pd.read_parquet(p)
    value_cols = [c for c in ["step", "distance", "speed", "calories", "running"] if c in df.columns]
    df = prep_sensor_df(df, value_cols)

    agg_dict = {}
    if "step" in df.columns:
        agg_dict["step"] = ["sum", "mean"]
    if "distance" in df.columns:
        agg_dict["distance"] = ["sum", "mean"]
    if "speed" in df.columns:
        agg_dict["speed"] = ["mean", "max"]
    if "calories" in df.columns:
        agg_dict["calories"] = ["sum", "mean"]
    if "running" in df.columns:
        agg_dict["running"] = ["sum", "mean"]

    feat = df.groupby(["subject_id", "lifelog_date"]).agg(agg_dict).reset_index()
    feat.columns = ["subject_id", "lifelog_date"] + [
        f"{a}_{b}" for a, b in feat.columns.tolist()[2:]
    ]
    return feat

pedo_feat = build_pedo_features()
print(None if pedo_feat is None else pedo_feat.shape)
display(pedo_feat.head() if pedo_feat is not None else pd.DataFrame())

In [ ]:
# Cell 10
feature_tables = [activity_feat, light_feat, screen_feat, charge_feat, hr_feat, pedo_feat]

feat = base_df.copy()
for ft in feature_tables:
    if ft is not None:
        feat = feat.merge(ft, on=["subject_id", "lifelog_date"], how="left")

print(feat.shape)
display(feat.head())

In [ ]:
# Cell 11
stable_candidates = [
    c for c in feat.columns
    if c not in ["subject_id", "lifelog_date"] + TARGETS
]

core_personal_cols = [
    c for c in stable_candidates
    if c in [
        "heart_rate_mean", "heart_rate_std", "heart_rate_sleep_mean", "heart_rate_active_mean",
        "step_sum", "distance_sum", "speed_mean",
        "m_screen_use_mean", "m_light_mean", "m_activity_mean"
    ]
]

print("stable feature count:", len(stable_candidates))
print("core personal:", core_personal_cols)

In [ ]:
# Cell 12
def add_prior_features(df, cols):
    df = df.sort_values(["subject_id", "lifelog_date"]).copy()

    for col in cols:
        grp = df.groupby("subject_id")[col]

        prior_mean = grp.expanding().mean().shift(1).reset_index(level=0, drop=True)
        prior_std = grp.expanding().std().shift(1).reset_index(level=0, drop=True)
        prior_cnt = df.groupby("subject_id").cumcount()

        df[f"{col}_prior_mean"] = prior_mean
        df[f"{col}_prior_std"] = prior_std
        df[f"{col}_prior_cnt"] = prior_cnt
        df[f"{col}_dev"] = df[col] - df[f"{col}_prior_mean"]

    return df

feat = add_prior_features(feat, core_personal_cols)
display(feat.head())

In [ ]:
# Cell 13
train_mask = feat[TARGETS[0]].notnull()

global_means = feat.loc[train_mask, stable_candidates].mean(numeric_only=True)
global_stds = feat.loc[train_mask, stable_candidates].std(numeric_only=True)

for col in core_personal_cols:
    feat[f"{col}_prior_mean"] = feat[f"{col}_prior_mean"].fillna(global_means.get(col, 0.0))
    feat[f"{col}_prior_std"] = feat[f"{col}_prior_std"].fillna(global_stds.get(col, 1.0))
    feat[f"{col}_dev"] = feat[f"{col}_dev"].fillna(0.0)

final_features = stable_candidates + sum(
    [[f"{c}_prior_mean", f"{c}_prior_std", f"{c}_prior_cnt", f"{c}_dev"] for c in core_personal_cols],
    []
)

final_features = [c for c in final_features if c in feat.columns]
print(len(final_features))

In [ ]:
# Cell 14
train_feat = feat[feat[TARGETS[0]].notnull()].copy().reset_index(drop=True)
test_feat = feat[feat[TARGETS[0]].isnull()].copy().reset_index(drop=True)

X_train = train_feat[final_features].copy()
X_test = test_feat[final_features].copy()

med = X_train.median(numeric_only=True)
X_train = X_train.fillna(med)
X_test = X_test.fillna(med)

print(train_feat.shape, test_feat.shape, X_train.shape, X_test.shape)

In [ ]:
# Cell 15
def build_global_time_split(df, valid_ratio=0.2):
    uniq_dates = np.sort(df["lifelog_date"].unique())
    n_valid = max(1, int(len(uniq_dates) * valid_ratio))
    valid_dates = set(uniq_dates[-n_valid:])
    tr_idx = df.index[~df["lifelog_date"].isin(valid_dates)].tolist()
    va_idx = df.index[df["lifelog_date"].isin(valid_dates)].tolist()
    return tr_idx, va_idx

tr_idx, va_idx = build_global_time_split(train_feat, valid_ratio=0.2)
print(len(tr_idx), len(va_idx))
print(train_feat.loc[va_idx, "lifelog_date"].min(), train_feat.loc[va_idx, "lifelog_date"].max())

In [ ]:
# Cell 16
def avg_logloss(y_true_df, pred_df):
    scores = []
    for t in TARGETS:
        y_true = y_true_df[t].astype(int).values
        y_pred = np.clip(pred_df[t].values, 1e-6, 1 - 1e-6)
        scores.append(log_loss(y_true, y_pred))
    return float(np.mean(scores)), scores

def shrink_proba(p, alpha=0.8):
    return np.clip(0.5 + alpha * (p - 0.5), 1e-6, 1 - 1e-6)

def clip_proba(p, lo=0.03, hi=0.97):
    return np.clip(p, lo, hi)

In [ ]:
# Cell 17
def fit_predict_split(X_train, y_df, X_test, tr_idx, va_idx, features):
    oof_lgb = pd.DataFrame(index=va_idx, columns=TARGETS, dtype=float)
    oof_cat = pd.DataFrame(index=va_idx, columns=TARGETS, dtype=float)
    test_lgb = pd.DataFrame(index=X_test.index, columns=TARGETS, dtype=float)
    test_cat = pd.DataFrame(index=X_test.index, columns=TARGETS, dtype=float)

    X_tr = X_train.loc[tr_idx, features]
    X_va = X_train.loc[va_idx, features]
    X_te = X_test[features]

    for t in TARGETS:
        y_tr = y_df.loc[tr_idx, t].astype(int)
        y_va = y_df.loc[va_idx, t].astype(int)

        lgb_model = lgb.LGBMClassifier(
            n_estimators=300,
            learning_rate=0.03,
            num_leaves=15,
            max_depth=4,
            min_child_samples=20,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=1.0,
            reg_lambda=3.0,
            objective="binary",
            class_weight="balanced",
            random_state=42
        )
        lgb_model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="binary_logloss",
            callbacks=[lgb.early_stopping(50, verbose=False)]
        )

        cat_model = CatBoostClassifier(
            iterations=300,
            learning_rate=0.03,
            depth=4,
            loss_function="Logloss",
            eval_metric="Logloss",
            random_seed=42,
            verbose=False
        )
        cat_model.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=False)

        oof_lgb.loc[va_idx, t] = lgb_model.predict_proba(X_va)[:, 1]
        oof_cat.loc[va_idx, t] = cat_model.predict_proba(X_va)[:, 1]

        test_lgb[t] = lgb_model.predict_proba(X_te)[:, 1]
        test_cat[t] = cat_model.predict_proba(X_te)[:, 1]

    return oof_lgb, oof_cat, test_lgb, test_cat

oof_lgb, oof_cat, test_lgb, test_cat = fit_predict_split(
    X_train, train_feat[TARGETS], X_test, tr_idx, va_idx, final_features
)

pred_avg = 0.5 * oof_lgb + 0.5 * oof_cat

print("LGB :", avg_logloss(train_feat.loc[va_idx, TARGETS], oof_lgb))
print("CAT :", avg_logloss(train_feat.loc[va_idx, TARGETS], oof_cat))
print("AVG :", avg_logloss(train_feat.loc[va_idx, TARGETS], pred_avg))

In [ ]:
# Cell 18
rows = []
for alpha in [1.0, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.65, 0.60]:
    tmp = pred_avg.copy()
    for t in TARGETS:
        tmp[t] = clip_proba(shrink_proba(tmp[t].values, alpha=alpha), 0.03, 0.97)
    score, each = avg_logloss(train_feat.loc[va_idx, TARGETS], tmp)
    rows.append([alpha, score] + each)

alpha_df = pd.DataFrame(rows, columns=["alpha", "avg_logloss"] + TARGETS).sort_values("avg_logloss")
display(alpha_df)

best_alpha = float(alpha_df.iloc[0]["alpha"])
print("best_alpha =", best_alpha)

In [ ]:
# Cell 19
best_target_model = {}
for t in TARGETS:
    lgb_score = log_loss(train_feat.loc[va_idx, t].astype(int), np.clip(oof_lgb[t], 1e-6, 1 - 1e-6))
    cat_score = log_loss(train_feat.loc[va_idx, t].astype(int), np.clip(oof_cat[t], 1e-6, 1 - 1e-6))
    best_target_model[t] = "lgb" if lgb_score <= cat_score else "cat"

best_target_model

In [ ]:
# Cell 20
final_pred_lgb = pd.DataFrame(index=X_test.index, columns=TARGETS, dtype=float)
final_pred_cat = pd.DataFrame(index=X_test.index, columns=TARGETS, dtype=float)

for t in TARGETS:
    y = train_feat[t].astype(int)

    lgb_model = lgb.LGBMClassifier(
        n_estimators=220,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=4,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1.0,
        reg_lambda=3.0,
        objective="binary",
        class_weight="balanced",
        random_state=42
    )
    lgb_model.fit(X_train[final_features], y)
    final_pred_lgb[t] = lgb_model.predict_proba(X_test[final_features])[:, 1]

    cat_model = CatBoostClassifier(
        iterations=220,
        learning_rate=0.03,
        depth=4,
        loss_function="Logloss",
        eval_metric="Logloss",
        random_seed=42,
        verbose=False
    )
    cat_model.fit(X_train[final_features], y, verbose=False)
    final_pred_cat[t] = cat_model.predict_proba(X_test[final_features])[:, 1]

display(final_pred_lgb.head())
display(final_pred_cat.head())

In [ ]:
# Cell 21
final_pred = pd.DataFrame(index=X_test.index, columns=TARGETS, dtype=float)

for t in TARGETS:
    p = final_pred_lgb[t].values if best_target_model[t] == "lgb" else final_pred_cat[t].values
    p = shrink_proba(p, alpha=best_alpha)
    p = clip_proba(p, 0.03, 0.97)
    final_pred[t] = p

display(final_pred.describe().T)

In [ ]:
# Cell 22
submission = sub.copy()

for t in TARGETS:
    submission[t] = final_pred[t].values

save_dir = BASE_DIR / "submissions"
save_dir.mkdir(parents=True, exist_ok=True)

save_path = save_dir / "sub_stable_time_personal_lgbcat.csv"
submission.to_csv(save_path, index=False)

print(save_path)
display(submission.head())

In [ ]:
# Cell 23
# 개인화 제거 버전 빠른 비교용

no_personal_features = [c for c in stable_candidates if c in X_train.columns]

oof_lgb_np, oof_cat_np, _, _ = fit_predict_split(
    X_train[no_personal_features],
    train_feat[TARGETS],
    X_test[no_personal_features],
    tr_idx,
    va_idx,
    no_personal_features
)

pred_avg_np = 0.5 * oof_lgb_np + 0.5 * oof_cat_np
print("NO_PERSONAL AVG :", avg_logloss(train_feat.loc[va_idx, TARGETS], pred_avg_np))

In [ ]:
# Cell 24
# 제출 전 확률 분포 확인

for t in TARGETS:
    print(
        t,
        "mean=", round(submission[t].mean(), 4),
        "std=", round(submission[t].std(), 4),
        "min=", round(submission[t].min(), 4),
        "max=", round(submission[t].max(), 4)
    )

In [ ]:
# Cell 25
# 저장용 feature dump

feature_dump_path = BASE_DIR / "artifacts_day_feature_table.csv"
feature_dump_path.parent.mkdir(parents=True, exist_ok=True)
feat.to_csv(feature_dump_path, index=False)
print(feature_dump_path)

In [ ]:
# Cell 26
# 필요하면 이 셀만 수정해서 실험
# 1) final_features = stable_candidates
# 2) final_features = stable_candidates + 개인화 일부
# 3) best_alpha 수동 고정
# 4) LightGBM/CatBoost 비중 조정

print("train rows:", len(train_feat))
print("test rows:", len(test_feat))
print("n features:", len(final_features))
print("best_alpha:", best_alpha)
print("best_target_model:", best_target_model)